# 01 — Esplorazione dei dati (SH17)
Conteggio immagini/annotazioni, distribuzione delle classi, dimensioni delle bounding box, sbilanciamento delle classi.
Esegui prima `scripts/prepare_dataset.py` in modo che `data/processed/sh17` esista.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from src.config import set_reproducibility, select_device, SH17_CLASSES
set_reproducibility(42); print('device:', select_device('auto'))

In [ ]:
from src.data import audit_dataset
from src.visualization import plot_class_distribution, plot_bbox_size_distribution

IMAGES_DIR = root / 'data/processed/sh17/images/train'
stats = audit_dataset(IMAGES_DIR, SH17_CLASSES)
{k: v for k, v in stats.items() if not k.startswith('_')}

In [ ]:
plot_class_distribution(stats['per_class_counts'], root / 'reports/figures/class_distribution.png')
plot_bbox_size_distribution(stats['_bbox_wh'], root / 'reports/figures/bbox_size_distribution.png')
print('saved figures to reports/figures/')

## Esempi di immagini annotate
Visualizza alcune immagini con le loro bounding box ground truth.

In [ ]:
import cv2, random
from src.data import list_images, read_yolo_label, label_path_for_image
from src.visualization import draw_detections
import matplotlib.pyplot as plt

imgs = list_images(IMAGES_DIR)[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, p in zip(axes.ravel(), imgs):
    im = cv2.imread(str(p)); h, w = im.shape[:2]
    dets = [{'cls': b.cls, 'conf': 1.0, 'xyxy': b.to_xyxy(w, h)} for b in read_yolo_label(label_path_for_image(p))]
    ax.imshow(cv2.cvtColor(draw_detections(im, dets, SH17_CLASSES), cv2.COLOR_BGR2RGB)); ax.axis('off')
fig.savefig(root / 'reports/figures/sample_annotations.png', dpi=120); plt.show()